In [52]:
import pandas as pd
from ydata_profiling import ProfileReport

In [53]:
features = pd.read_csv("../data/raw/raw_features.csv", index_col=0)
metadata_cols = pd.read_csv("../data/processed/metadata_filtrada.csv").set_index("sample")[["Species", "WHO_Priority"]]

In [54]:
df_ydata = features.join(metadata_cols)

df_ydata.head()

,mdr_score_cromossomal,n_class_A_carbapenemase_cromossomal,n_class_A_carbapenemase_plasmidial,n_class_B_mbl_cromossomal,n_class_B_mbl_plasmidial,n_class_C_ampc_cromossomal,n_class_C_ampc_plasmidial,n_class_D_oxa_cromossomal,n_class_D_oxa_plasmidial,n_esbl_cromossomal,...,has_mcr_colistin_plasmidial,has_aminoglycoside_panres_cromossomal,has_aminoglycoside_panres_plasmidial,prop_carbapenemase_BD_plasmidial,pct_contigs_plasmidial,n_plasmids_com_conjugacao,gc_diff_plasmid_cromossomo,virulence_score_cromossomal,Species,WHO_Priority
sample,,,,,,,,,,,,,,,,,,,,,
GCA_000216055.2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0.0,1.655629,0,0.18,0,Leptospira interrogans,Other
GCA_000316425.1,13,0,0,0,0,1,0,0,0,0,...,0,0,0,0.0,5.675676,5,-2.92,8,Escherichia coli,Critical
GCA_000223095.2,7,0,0,1,0,0,0,0,0,0,...,0,0,0,0.0,6.349206,0,-3.05,6,Vibrio cholerae,Other
GCA_036761135.1,10,0,0,0,0,0,0,1,1,0,...,0,0,0,0.5,18.269231,2,0.31,4,Acinetobacter bereziniae,Other
GCA_003670255.1,8,0,0,0,0,0,0,0,2,0,...,0,0,0,1.0,22.406639,9,0.90,4,Acinetobacter bereziniae,Other


In [55]:
colunas = ['n_class_A_carbapenemase_plasmidial', 'n_class_B_mbl_plasmidial', 'n_class_C_ampc_plasmidial', 
 'n_class_D_oxa_plasmidial', 'n_esbl_plasmidial', 'n_genes_efflux_plasmidial', 'pct_contigs_plasmidial', 
 'n_plasmids_com_conjugacao', 'gc_diff_plasmid_cromossomo', 'Species', 'WHO_Priority']

In [56]:
df_ydata_critical = df_ydata[df_ydata['WHO_Priority'] == 'Critical']
df_ydata_critical = df_ydata_critical[colunas]

## Profile antes

In [57]:
profile = ProfileReport(df_ydata_critical, title="Features (pré-estruturação)", minimal=False)
profile.to_file("../reports/features_antes_ydata_critical.html")

Summarize dataset:   0%|          | 0/16 [00:00<?, ?it/s, Describe variable: n_esbl_plasmidial]                 

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 80.91it/s]


## Estruturação

Usando o `df_ydata_critical` e as colunas: ['n_class_A_carbapenemase_plasmidial', 'n_class_B_mbl_plasmidial',
       'n_class_C_ampc_plasmidial', 'n_class_D_oxa_plasmidial',
       'n_esbl_plasmidial', 'n_genes_efflux_plasmidial',
       'pct_contigs_plasmidial', 'n_plasmids_com_conjugacao',
       'gc_diff_plasmid_cromossomo', 'Species', 'WHO_Priority']

#### Correlação:
- n_plasmids_com_conjugacao <-> pct_contigs_plasmidial
- n_esbl_plasmidial  <-> pct_contigs_plasmidial
- *Solução: remover o pct_contigs_plasmidial* 

#### Duplicados:
- 23 linhas duplicadas
- *Verificar o que são esses genomas*

#### Imbalanceado:
- n_class_B_mbl_plasmidial (de 2230, 2039 são 0)
- n_class_C_ampc_plasmidial (de 2230, 2039 são 0)
- n_class_D_oxa_plasmidial

#### `gc_diff_plasmid_cromossomo`:
- imputação

In [58]:
colunas = ['n_class_A_carbapenemase_plasmidial', 'prop_carbapenemase_BD_plasmidial', 'n_esbl_plasmidial', 'n_genes_efflux_plasmidial', 'pct_contigs_plasmidial', 
 'n_plasmids_com_conjugacao', 'gc_diff_plasmid_cromossomo', 'Species', 'WHO_Priority']

df_ydata_critical = df_ydata[df_ydata['WHO_Priority'] == 'Critical']
df_ydata_critical = df_ydata_critical[colunas]

### 1. gc_diff_plasmid_cromossomo

In [59]:
print("n. de genomas sem plasmídeo, com NaN:", df_ydata_critical['gc_diff_plasmid_cromossomo'].isna().sum())
df_ydata_critical['gc_diff_plasmid_cromossomo'].describe()

n. de genomas sem plasmídeo, com NaN: 13


count    2217.000000
mean       -1.420302
std         1.736464
min       -18.780000
25%        -2.040000
50%        -1.160000
75%        -0.390000
max        12.950000
Name: gc_diff_plasmid_cromossomo, dtype: float64

In [60]:
q1 = df_ydata_critical['gc_diff_plasmid_cromossomo'].quantile(0.25)
q3 = df_ydata_critical['gc_diff_plasmid_cromossomo'].quantile(0.75)

mask = df_ydata_critical['gc_diff_plasmid_cromossomo'].isna()
mask.head()

sample
GCA_000316425.1    False
GCA_000355215.1    False
GCA_000355195.1    False
GCA_000355235.1    False
GCA_000355255.1    False
Name: gc_diff_plasmid_cromossomo, dtype: bool

In [61]:
import numpy as np

In [62]:
df_ydata_critical.loc[mask, 'gc_diff_plasmid_cromossomo'] = np.random.uniform(
    q1, q3, size=mask.sum()
)

In [63]:
df_ydata_critical[df_ydata_critical['pct_contigs_plasmidial'] == 0]['gc_diff_plasmid_cromossomo'].head()

sample
GCF_001649555.1   -1.693982
GCA_002895205.1   -0.511083
GCA_022660815.1   -0.538646
GCA_002887715.1   -1.672680
GCA_002900305.1   -1.454476
Name: gc_diff_plasmid_cromossomo, dtype: float64

### 2. Correlação

Como no ydata mostrava correlaçao, mas aqui vejo que a unica correlação é de ~50%, vou manter

In [64]:
corr = df_ydata_critical.select_dtypes(include='number').corr()
limite = 0.5
corr[corr > limite].style.background_gradient(cmap='viridis')

,n_class_A_carbapenemase_plasmidial,prop_carbapenemase_BD_plasmidial,n_esbl_plasmidial,n_genes_efflux_plasmidial,pct_contigs_plasmidial,n_plasmids_com_conjugacao,gc_diff_plasmid_cromossomo
n_class_A_carbapenemase_plasmidial,1.000000,nan,0.500645,nan,nan,nan,nan
prop_carbapenemase_BD_plasmidial,nan,1.000000,nan,nan,nan,nan,nan
n_esbl_plasmidial,0.500645,nan,1.000000,nan,nan,nan,nan
n_genes_efflux_plasmidial,nan,nan,nan,1.000000,nan,nan,nan
pct_contigs_plasmidial,nan,nan,nan,nan,1.000000,nan,nan
n_plasmids_com_conjugacao,nan,nan,nan,nan,nan,1.000000,nan
gc_diff_plasmid_cromossomo,nan,nan,nan,nan,nan,nan,1.000000


### 3. Duplicadas

In [65]:
duplicadas = df_ydata_critical[df_ydata_critical.duplicated(keep=False)].sort_values(by='gc_diff_plasmid_cromossomo')

duplicadas.head()

,n_class_A_carbapenemase_plasmidial,prop_carbapenemase_BD_plasmidial,n_esbl_plasmidial,n_genes_efflux_plasmidial,pct_contigs_plasmidial,n_plasmids_com_conjugacao,gc_diff_plasmid_cromossomo,Species,WHO_Priority
sample,,,,,,,,,
GCA_041221925.1,1,0.0,2,1,80.0,4,-5.73,Klebsiella pneumoniae,Critical
GCA_041212775.1,1,0.0,2,1,80.0,4,-5.73,Klebsiella pneumoniae,Critical
GCA_041222175.1,1,0.0,3,3,80.0,4,-5.55,Klebsiella pneumoniae,Critical
GCA_041221345.1,1,0.0,3,3,80.0,4,-5.55,Klebsiella pneumoniae,Critical
GCA_029079765.1,1,0.0,3,2,80.0,4,-5.48,Klebsiella pneumoniae,Critical


In [66]:
df_ydata_critical = df_ydata_critical.drop_duplicates(keep='first')

duplicadas = df_ydata_critical[df_ydata_critical.duplicated(keep=False)].sort_values(by='gc_diff_plasmid_cromossomo')

duplicadas.head()


,n_class_A_carbapenemase_plasmidial,prop_carbapenemase_BD_plasmidial,n_esbl_plasmidial,n_genes_efflux_plasmidial,pct_contigs_plasmidial,n_plasmids_com_conjugacao,gc_diff_plasmid_cromossomo,Species,WHO_Priority
sample,,,,,,,,,


### 4. Imbalanceados
Voltei no notebook anterior, criei a coluna 'prop_carbapensemase_BD_plasmidial', que corresponde a % dos genes que conferem resistencia a carbapenemicos (classe B e D) em plasmídio (plasmidio/total)

In [68]:
df_ydata_critical

,n_class_A_carbapenemase_plasmidial,prop_carbapenemase_BD_plasmidial,n_esbl_plasmidial,n_genes_efflux_plasmidial,pct_contigs_plasmidial,n_plasmids_com_conjugacao,gc_diff_plasmid_cromossomo,Species,WHO_Priority
sample,,,,,,,,,
GCA_000316425.1,0,0.0,0,2,5.675676,5,-2.92,Escherichia coli,Critical
GCA_000355215.1,0,0.0,0,10,42.857143,11,-2.10,Escherichia coli,Critical
GCA_000355195.1,0,0.0,0,3,44.230769,8,-1.50,Escherichia coli,Critical
GCA_000355235.1,0,0.0,1,2,9.677419,2,-1.13,Escherichia coli,Critical
GCA_000355255.1,0,0.0,1,2,33.333333,10,-1.17,Escherichia coli,Critical
...,...,...,...,...,...,...,...,...,...
GCA_054551815.1,1,0.0,0,1,60.000000,2,-2.28,Enterobacter roggenkampii,Critical
GCA_054551855.1,1,0.0,0,1,66.666667,2,-0.03,Enterobacter roggenkampii,Critical
GCA_054551875.1,1,0.0,0,1,28.571429,2,-0.42,Enterobacter roggenkampii,Critical


## Profile depois

In [69]:
profile_depois = ProfileReport(df_ydata_critical, title="Features (pós-estruturação)", minimal=False)
profile_depois.to_file("../reports/features_depois_ydata_critical.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 63.41it/s]


In [78]:
df_ydata_critical['prop_carbapenemase_BD_plasmidial'] = (df_ydata_critical['prop_carbapenemase_BD_plasmidial'] > 0).astype(int)

In [80]:
df_ydata_critical['prop_carbapenemase_BD_plasmidial'].value_counts()

prop_carbapenemase_BD_plasmidial
0    1916
1     294
Name: count, dtype: int64

In [81]:
profile_depois = ProfileReport(df_ydata_critical, title="Features (pós-estruturação)", minimal=False)
profile_depois.to_file("../reports/features_depois_ydata_critical.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 144.46it/s]


In [86]:
df_ydata_critical.to_csv("../data/processed/features.csv")